In [1]:
import os 
import pandas as pd
import numpy as np
import glob
pd.set_option('display.max_columns', None)

In [2]:
#ADNI dataset 
#"/project_cephfs/3022017.06/ADNI/phenotypes/Velocity files" --> may contain this. Also the surface holes = euler nr
root = "/project_cephfs/3022017.06/ADNI/freesurfer"

#aseg data
aseg_path = os.path.join(root, "aseg_stats.txt")
aseg_data = pd.read_csv(
    aseg_path,
    sep="\\s+",
    engine='python'
)

#lh data
lh_path = os.path.join(root, "lh_aparc_a2009s_stats.txt")
lh_data = pd.read_csv(
    lh_path,
    sep="\\s+",
    engine='python'
)

#rh data
rh_path = os.path.join(root, "rh_aparc_a2009s_stats.txt")
rh_data = pd.read_csv(
    rh_path,
    sep="\\s+",
    engine='python'
)

#merge left and right
lh_data = lh_data.rename(columns={"lh.aparc.a2009s.thickness" : "scan_id"})
rh_data = rh_data.rename(columns={"rh.aparc.a2009s.thickness" : "scan_id"})
lh_rh_data = pd.merge(lh_data, rh_data, on="scan_id", how="left")

#merge lh_rh and aseg
aseg_data = aseg_data.rename(columns={"Measure:volume" : "scan_id"})
adni_data = pd.merge(lh_rh_data, aseg_data, on="scan_id", how="left")
adni_data

#removing all the 'long' files 
adni_data = adni_data[~adni_data['scan_id'].str.contains('long', case=False, na=False)]

#replacing the full name with the subj nr to be able to merge with the covariates dataset
adni_data["scan_id"] = adni_data["scan_id"].str.replace(r'^.*?(?=I\d+$)', '', regex=True)


In [3]:
#age/sex/site data
root = "/project_cephfs/3022017.06/ADNI/phenotypes"
asl_path = os.path.join(root, "Velocity_4_04_2025.csv")
asl_data = pd.read_csv(
    asl_path,
    sep=",",
    engine='python'
)

#age/sex/site data
root = "/project_cephfs/3022017.06/ADNI/phenotypes"
asl_path2 = os.path.join(root, "Velocity_MP-RAGE_4_04_2025.csv")
asl_data2 = pd.read_csv(
    asl_path2,
    sep=",",
    engine='python'
)

asl_data = asl_data[['Image Data ID', 'Subject','Group', 'Sex', 'Age']]
asl_data2 = asl_data2[['Image Data ID', 'Subject','Group', 'Sex', 'Age']]

cov_data = pd.concat([asl_data, asl_data2], axis=0)
cov_data = cov_data.rename(columns={"Image Data ID" : "scan_id"})
merged = cov_data.merge(adni_data, on="scan_id", how="inner")



In [4]:
root = "/project_cephfs/3022017.06/ADNI/phenotypes"
cov_path = os.path.join(root, "JD_covar_all_subjects_SV.xlsx")
df_cov = pd.read_excel(cov_path)

In [5]:
root = "/project_cephfs/3022017.06/ADNI/phenotypes"
"""
ADNIMERGE_02May2025.csv
All_Subjects_DXSUM_02May2025.csv
MRI3META_19May2025.csv
MRIMETA_19May2025.csv

"""
cov_path = os.path.join(root, "ADNIMERGE_02May2025.csv")
df_cov = pd.read_csv(cov_path)
df_cov = df_cov[["SITE", "PTID", "AGE", "PTGENDER"]]

/scratch/quirom/slurm_job_51193262/ipykernel_747199/3492745866.py:10: DtypeWarning: Columns (19,20,21,50,51,104,105,106) have mixed types. Specify dtype option on import or set low_memory=False.
  df_cov = pd.read_csv(cov_path)


In [6]:
#OASIS2 data
root = "/project_cephfs/3022017.06/OASIS2"

#aseg data
aseg_path = os.path.join(root, "freesurfer","aseg_stats_all.txt")
aseg_data = pd.read_csv(
    aseg_path,
    sep="\\s+",
    engine='python'
)

#lh data
lh_path = os.path.join(root, "freesurfer","lh_aparc_a2009s.txt")
lh_data = pd.read_csv(
    lh_path,
    sep="\\s+",
    engine='python'
)

#rh data
rh_path = os.path.join(root,"freesurfer", "rh_aparc_a2009s.txt")
rh_data = pd.read_csv(
    rh_path,
    sep="\\s+",
    engine='python'
)

#euler nr
euler_path = os.path.join(root, "freesurfer","euler_number.csv")
euler_data = pd.read_csv(
    euler_path,
    sep=",",
    engine='python'
)

#demographic data
demo_path = os.path.join(root, "phenotypes","oasis_longitudinal_demographics.csv")
demo_data = pd.read_csv(
    demo_path,
    sep=",",
    engine='python'
)

#rename indices
lh_data = lh_data.rename(columns={"lh.aparc.a2009s.thickness" : "scan_id"})
rh_data = rh_data.rename(columns={"rh.aparc.a2009s.thickness" : "scan_id"})
aseg_data = aseg_data.rename(columns={"Measure:volume" : "scan_id"})
euler_data = euler_data.rename(columns={"Unnamed: 0": "scan_id"})
demo_data = demo_data.rename(columns={"participant_id": "scan_id"})
demo_data = demo_data.drop(columns=["matching_id","Visit_id"])

#merge lh and rh               
lh_rh_data = pd.merge(lh_data, rh_data, on="scan_id", how="left")

#merge lh_rh and aseg
aseg_lhrh_data = pd.merge(lh_rh_data, aseg_data, on="scan_id", how="left")

#merge aseg_lhrh with euler data
aseg_eu = pd.merge(aseg_lhrh_data, euler_data, on="scan_id", how="left")
oasis2_data = pd.merge(aseg_eu, demo_data, on="scan_id", how="left")

In [7]:
#OASIS3 data
root = "/project_cephfs/3022017.06/OASIS3"

#aseg data
aseg_path = os.path.join(root, "freesurfer","aseg_stats_all.txt")
aseg_data = pd.read_csv(
    aseg_path,
    sep="\\s+",
    engine='python'
)

#lh data
lh_path = os.path.join(root, "freesurfer","lh_aparc_a2009s.txt")
lh_data = pd.read_csv(
    lh_path,
    sep="\\s+",
    engine='python'
)

#rh data
rh_path = os.path.join(root,"freesurfer", "rh_aparc_a2009s.txt")
rh_data = pd.read_csv(
    rh_path,
    sep="\\s+",
    engine='python'
)

#euler nr
euler_path = os.path.join(root, "freesurfer","euler_number.csv")
euler_data = pd.read_csv(
    euler_path,
    sep=",",
    engine='python'
)

#demographic data
demo_path = os.path.join(root, "docs","covariates.csv")
demo_data = pd.read_csv(
    demo_path,
    sep=",",
    engine='python'
)


#rename indices
lh_data = lh_data.rename(columns={"lh.aparc.a2009s.thickness" : "scan_id"})
rh_data = rh_data.rename(columns={"rh.aparc.a2009s.thickness" : "scan_id"})
aseg_data = aseg_data.rename(columns={"Measure:volume" : "scan_id"})
euler_data = euler_data.rename(columns={"Unnamed: 0": "scan_id"})
demo_data = demo_data.rename(columns={"participant_id":"scan_id"})


#merge lh and rh               
lh_rh_data = pd.merge(lh_data, rh_data, on="scan_id", how="left")

#merge lh_rh and aseg
aseg_lhrh_data = pd.merge(lh_rh_data, aseg_data, on="scan_id", how="left")

#merge aseg_lhrh with euler data
aseg_eu = pd.merge(aseg_lhrh_data, euler_data, on="scan_id", how="left")
oasis3_data = pd.merge(aseg_eu, demo_data, on="scan_id", how="left")

# this didnt work


In [8]:
# #OASIS3 data
# root = "/project_cephfs/3022017.06/OASIS3"

# #aseg data
# aseg_path = os.path.join(root, "freesurfer","aseg_stats_all.txt")
# aseg_data = pd.read_csv(
#     aseg_path,
#     sep="\\s+",
#     engine='python'
# )

# #lh data
# lh_path = os.path.join(root, "freesurfer","lh_aparc_a2009s.txt")
# lh_data = pd.read_csv(
#     lh_path,
#     sep="\\s+",
#     engine='python'
# )

# #rh data
# rh_path = os.path.join(root,"freesurfer", "rh_aparc_a2009s.txt")
# rh_data = pd.read_csv(
#     rh_path,
#     sep="\\s+",
#     engine='python'
# )

# #euler nr
# euler_path = os.path.join(root, "freesurfer","euler_number.csv")
# euler_data = pd.read_csv(
#     euler_path,
#     sep=",",
#     engine='python'
# )

# #demographic data
# demo_path = os.path.join(root, "docs","covariates.csv")
# demo_data = pd.read_csv(
#     demo_path,
#     sep=",",
#     engine='python'
# )

# #demographic data
# health_outcomes_path = os.path.join(root, "docs","Demographics.csv")
# health_outcomes_data = pd.read_csv(
#     health_outcomes_path,
#     sep=",",
#     engine='python'
# )

# #rename indices
# lh_data = lh_data.rename(columns={"lh.aparc.a2009s.thickness" : "scan_id"})
# rh_data = rh_data.rename(columns={"rh.aparc.a2009s.thickness" : "scan_id"})
# aseg_data = aseg_data.rename(columns={"Measure:volume" : "scan_id"})
# euler_data = euler_data.rename(columns={"Unnamed: 0": "scan_id"})
# demo_data = demo_data.rename(columns={"participant_id":"scan_id"})
# health_outcomes_data = health_outcomes_data.rename(columns={"ADRC_ADRCCLINICALDATA ID":"scan_id"})

# health_outcomes_data['scan_id'] = health_outcomes_data['scan_id'].str.replace('_ClinicalData_', '_MR_', regex=False)
# health_outcomes_data = health_outcomes_data[["scan_id" , "dx1"]].copy()


# #merge lh and rh               
# lh_rh_data = pd.merge(lh_data, rh_data, on="scan_id", how="left")

# #merge lh_rh and aseg
# aseg_lhrh_data = pd.merge(lh_rh_data, aseg_data, on="scan_id", how="left")

# #merge aseg_lhrh with euler data
# aseg_eu = pd.merge(aseg_lhrh_data, euler_data, on="scan_id", how="left")

# #merge health_outcomes_data with aseg_eu
# aseg_eu_hc = pd.merge(aseg_eu, health_outcomes_data, on="scan_id", how="left")

# #final data
# oasis3_data = pd.merge(aseg_eu_hc, demo_data, on="scan_id", how="left")


# oasis3_data

# euler nr calculation for oasis3 data

In [9]:
#Andre advised me to use OASIS3 

#now lets filter based on the calculation of the euler number
oasis3_data = oasis3_data[oasis3_data["site_id"] > 0]
oasis3_data = oasis3_data.astype({"scan_id": str, "lh_en": int, "rh_en": int, "site_id": int})

site_medians = oasis3_data.groupby("site_id")["avg_en"].median()
oasis3_data["site_id_median"] = oasis3_data["site_id"].map(site_medians)

oasis3_data["avg_en_centered"] = oasis3_data["avg_en"] - oasis3_data["site_id_median"]

oasis3_data["avg_en_centered_neg"] = -1 * oasis3_data["avg_en_centered"]
oasis3_data["avg_en_centered_neg_sqrt"] = np.sqrt(np.abs(oasis3_data["avg_en_centered_neg"]))
good_subj_df = oasis3_data[oasis3_data["avg_en_centered_neg_sqrt"] < 10]
bad_subj_df = oasis3_data[oasis3_data["avg_en_centered_neg_sqrt"] > 10]


#the total cortex volume is greater in the diagnosis == 0 group --> must be healthy control

alzheimer = good_subj_df[good_subj_df["diagnosis"] == 1.0]
healthy_controls = good_subj_df[good_subj_df["diagnosis"] == 0.0]
unknown = good_subj_df[good_subj_df["diagnosis"] == 2.0]

vol_ad = alzheimer["CortexVol"].mean()
vol_hc = healthy_controls["CortexVol"].mean()
vol_un = unknown["CortexVol"].mean()

print(f"Volume of Alzheimer: {vol_ad:.2f}")
print(f"Volume of healthy control: {vol_hc:.2f}")
print(f"Volume of unknown: {vol_un:.2f}")

oasis3_df = good_subj_df.copy()
oasis3_df = oasis3_df.rename(columns={"scan_id":"SubjectID","age":"Age","gender":"Sex","site_id":"Site"})

Volume of Alzheimer: 394440.67
Volume of healthy control: 414691.87
Volume of unknown: 407258.23


/scratch/quirom/slurm_job_51193262/ipykernel_747199/1761858174.py:8: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  oasis3_data["site_id_median"] = oasis3_data["site_id"].map(site_medians)
/scratch/quirom/slurm_job_51193262/ipykernel_747199/1761858174.py:10: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  oasis3_data["avg_en_centered"] = oasis3_data["avg_en"] - oasis3_data["site_id_median"]
/scratch/quirom/slurm_job_51193262/ipykernel_747199/1761858174.py:12: PerformanceWarning: DataFrame is highly fragmented.  This is usually the 

In [13]:
ukb_df = pd.read_csv('/home/preclineu/quirom/Desktop/internship/Normative-Models-and-VAE/logs/final_data/Final_data.csv')
ukb_df = ukb_df.iloc[:,2:]
ukb_cols = ukb_df.columns


ukb_df['SubjectID'] = ukb_df['SubjectID'].astype(str)

oasis3_df_filtered = oasis3_df.filter(items=ukb_df.columns)
oasis3_df_filtered["Diagnosis"] = good_subj_df["diagnosis"].copy()
oasis3_df_filtered["Diagnosis"].count()

#"""https://theunitedconsortium.com/wp-content/uploads/2021/07/OASIS-3_Imaging_Data_Dictionary_v1.8.pdf"""


np.int64(1893)